Trabajo Intedrador

Preguntas Generadas (Hito 1)

¿De qué manera el nivel de ingresos familiares condiciona la tasa de retorno de las tutorías académicas?

¿En qué medida un nivel alto de involucramiento parental puede neutralizar el impacto negativo de una "Calidad Docente" baja o media, y es este efecto más determinante en escuelas públicas que en privadas?

¿En qué punto el incremento de horas de estudio comienza a mostrar rendimientos decrecientes debido a la privación de sueño o falta de actividad física?

In [36]:
#HITO 2 
import pandas as pd
import numpy as np 

In [37]:
df = pd.read_csv('StudentPerformanceFactors.csv')

#exploracion inicial
print("Dimensiones del dataset")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

print("\n Primeras 5 filas")
display(df.head())

print("\n Informacion de columnas")
print(df.info())

print("\n Descripcion")
print(df.describe())

Dimensiones del dataset
Filas: 6607, Columnas: 20

 Primeras 5 filas


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70



 Informacion de columnas
<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   str  
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   str  
 8   Internet_Access             6607 non-null   str  
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   str  
 11  Teacher_Quality             6529 non-null   str  
 12  School_Type                 6607 non-null   str  
 13  Peer_Influence              6607 non-null   str 

In [38]:
preguntas_investigacion = {
    "P1": "¿De qué manera el nivel de ingresos familiares condiciona la tasa de retorno de las tutorías académicas?",
    "P2": "¿En qué medida un nivel alto de involucramiento parental puede neutralizar el impacto negativo de una 'Calidad Docente' baja o media, y es este efecto más determinante en escuelas públicas que en privadas?",
    "P3": "¿En qué punto el incremento de horas de estudio comienza a mostrar rendimientos decrecientes debido a la privación de sueño o falta de actividad física?"
}

for clave, pregunta in preguntas_investigacion.items():
    print(f"{clave}: {pregunta}\n")

P1: ¿De qué manera el nivel de ingresos familiares condiciona la tasa de retorno de las tutorías académicas?

P2: ¿En qué medida un nivel alto de involucramiento parental puede neutralizar el impacto negativo de una 'Calidad Docente' baja o media, y es este efecto más determinante en escuelas públicas que en privadas?

P3: ¿En qué punto el incremento de horas de estudio comienza a mostrar rendimientos decrecientes debido a la privación de sueño o falta de actividad física?



In [39]:
#Deteccion y tratamieto de nulos 

#crear copia pata trabajar
df_limpio = df.copy()

#identificar culumnas con nulos
nulos = df_limpio.isnull().sum()
columnas_con_nulos = nulos[nulos > 0].index.tolist()

print(f"Columnas con valores nulos: {columnas_con_nulos}")

#Estrategia segun el tipo de dato
for col in columnas_con_nulos:
    if df_limpio[col].dtype != 'int64':
        #para categoricas se rellena con moda
        moda = df_limpio[col].mode()[0]
        df_limpio[col] = df_limpio[col].fillna(moda)
        print(f"{col}: Nulos rellenados con moda '{moda}'")
    else:
        #para numericos rellenar con mediana
        df_limpio[col] = pd.to_numeric(df_limpio[col], errors = 'coerce')
        mediana = df_limpio[col].median()
        df_limpio[col] = df_limpio[col].fillna(mediana)
        print(f"{col}: Nulos rellenados con mediana {mediana:.2f}")

#verificar que no queden nulos
#print(nulos)
print(f"\n Nulos restantes: {df_limpio.isnull().sum()}")


Columnas con valores nulos: ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']
Teacher_Quality: Nulos rellenados con moda 'Medium'
Parental_Education_Level: Nulos rellenados con moda 'High School'
Distance_from_Home: Nulos rellenados con moda 'Near'

 Nulos restantes: Hours_Studied                 0
Attendance                    0
Parental_Involvement          0
Access_to_Resources           0
Extracurricular_Activities    0
Sleep_Hours                   0
Previous_Scores               0
Motivation_Level              0
Internet_Access               0
Tutoring_Sessions             0
Family_Income                 0
Teacher_Quality               0
School_Type                   0
Peer_Influence                0
Physical_Activity             0
Learning_Disabilities         0
Parental_Education_Level      0
Distance_from_Home            0
Gender                        0
Exam_Score                    0
dtype: int64


In [40]:
#Deteccion y tratamiento de outliers
def eliminar_outliers_iqr(df, columnas_numericas,multiplicador = 1.5):
    #Elimina outliers usando el método IQR (Intervalo Interdecil) multiplier: 1.5 para outliers moderados, 3.0 para extremos
    df_sin_outliers = df.copy()
    filas_eliminadas = 0 

    for col in columnas_numericas:
        Q1 = df_sin_outliers[col].quantile(0.25)
        Q3 = df_sin_outliers[col].quantile(0.75)
        IQR = Q3 - Q1

        limite_inferior = Q1 - multiplicador * IQR
        limite_superior = Q3 + multiplicador * IQR

        #contar outlier antes de eliminar 
        outliers = ((df_sin_outliers[col] < limite_inferior) | 
                   (df_sin_outliers[col] > limite_superior)).sum()

        if outliers > 0:
            #filtrar los datos dentro de los limites 
            df_sin_outliers = df_sin_outliers[
                (df_sin_outliers[col] >= limite_inferior) & 
                (df_sin_outliers[col] <= limite_superior)
            ]

            filas_eliminadas += outliers
        print(f"{col}: {outliers} outliers eliminados[{limite_inferior:.2f}, {limite_superior:.2f}]")
    print(f"Dataset final: {len(df_sin_outliers)} filas")
    print(f"Total de filas borradas: {len(df) - len(df_sin_outliers)}")

    return df_sin_outliers

#aplicar eliminacion a columnas numericas
columnas_numericas = ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 
'Tutoring_Sessions', 'Physical_Activity', 'Exam_Score']
df_limpio = eliminar_outliers_iqr(df_limpio,columnas_numericas, multiplicador=3.0)

Hours_Studied: 0 outliers eliminados[-8.00, 48.00]
Attendance: 0 outliers eliminados[10.00, 150.00]
Sleep_Hours: 0 outliers eliminados[0.00, 14.00]


Previous_Scores: 0 outliers eliminados[-12.00, 163.00]
Tutoring_Sessions: 26 outliers eliminados[-2.00, 5.00]
Physical_Activity: 0 outliers eliminados[-4.00, 10.00]
Exam_Score: 43 outliers eliminados[53.00, 81.00]
Dataset final: 6538 filas
Total de filas borradas: 69


In [41]:
#Normalizacion de texto

def normalizar_columnas_texto (df, columnas):
    #estandarizar valores de texto (minusculas, sin espacios extras, titulos)
    df_normalizado = df.copy()

    for col in columnas:
        if df_normalizado[col].dtype == 'object':
            #convertir a string, lowercase. strip y title case
            df_normalizado[col] = (df_normalizado[col]
                                    .astype(str)
                                    .str.lower()
                                    .str.strip()
                                    .str.title())
            print(f"{col} normalizada")
    return df_normalizado

columnas_texto = ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities',
                  'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality',
                  'School_Type', 'Peer_Influence', 'Parental_Education_Level', 
                  'Distance_from_Home', 'Gender']

df_limpio = normalizar_columnas_texto(df_limpio, columnas_texto) 

#verificar normalizacion
print("\n Valores únicos después de normalización:")
for col in ['Parental_Involvement', 'School_Type', 'Gender']:
    print(f"{col}: {df_limpio[col].unique()}")


 Valores únicos después de normalización:
Parental_Involvement: <StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str
School_Type: <StringArray>
['Public', 'Private']
Length: 2, dtype: str
Gender: <StringArray>
['Male', 'Female']
Length: 2, dtype: str


In [42]:
#Feature engineering
df_final = df_limpio.copy()

#combinar asistencia y horas de estudio
df_final['indice_constancia'] = (
    (df_final['Attendance'] * 0.6) +
    (df_final['Hours_Studied'] * 10 * 0.4) #escalar horas para asimilar rango
)

#combinar recursos, internet y tutorias
df_final['score_recursos'] = (
    df_final['Access_to_Resources'].map({'Low': 1, 'Medium': 2, 'High': 3}) +
    df_final['Internet_Access'].map({'No': 0, 'Yes': 2}) +
    df_final['Tutoring_Sessions'] * 0.5
)

#crear categoria rendimiento
def categorizar_rendimiento(score):
    if score >= 80:
        return 'Alto'
    elif score >= 60:
        return 'Medio'
    else:
        return 'Bajo'

df_final['categoria_rendimiento'] = df_final['Exam_Score'].apply(categorizar_rendimiento)

#interaccion socioeconomica-escuela
df_final['interaccion_socioescuela'] = (
    df_final['Family_Income'].map({'Low': 1, 'Medium': 2, 'High': 3}) * 
    df_final['School_Type'].map({'Public': 1, 'Private': 2})
)

#balance vida-estudio
df_final['balance_vida_estudio'] = (
    df_final['Sleep_Hours'] + df_final['Physical_Activity']
) - (df_final['Hours_Studied'] * 0.5)

print("Nuevas features creadas:")
display(df_final[['indice_constancia', 'score_recursos', 'categoria_rendimiento', 
                'interaccion_socioescuela', 'balance_vida_estudio']].head())

Nuevas features creadas:


,indice_constancia,score_recursos,categoria_rendimiento,interaccion_socioescuela,balance_vida_estudio
0,142.4,5.0,Medio,1,-1.5
1,114.4,5.0,Medio,2,2.5
2,154.8,5.0,Medio,2,-1.0
3,169.4,4.5,Medio,2,-2.5
4,131.2,5.5,Medio,2,0.5


In [43]:
#Auditoria Final
print("AUDITORÍA FINAL DEL DATASET PROCESADO")
print("="*50)
print(f"Filas finales: {len(df_final)}")
print(f"Columnas totales: {len(df_final.columns)}")
print(f"Nulos restantes: {df_final.isnull().sum().sum()}")
print(f"Duplicados exactos: {df_final.duplicated().sum()}")

# Guardar dataset procesado
df_final.to_csv('StudentPerformanceFactors_procesado.csv', index=False)
print("\n Dataset guardado como 'StudentPerformanceFactors_procesado.csv'")

# Resumen de variables creadas
print("\n RESUMEN DE FEATURES CREADAS:")
features_nuevas = ['indice_constancia', 'score_recursos', 'categoria_rendimiento', 
                   'interaccion_socioescuela', 'balance_vida_estudio']
for feat in features_nuevas:
    print(f"• {feat}: {df_final[feat].dtype} - {df_final[feat].nunique()} valores únicos")

AUDITORÍA FINAL DEL DATASET PROCESADO
Filas finales: 6538
Columnas totales: 25
Nulos restantes: 0


Duplicados exactos: 0

 Dataset guardado como 'StudentPerformanceFactors_procesado.csv'

 RESUMEN DE FEATURES CREADAS:
• indice_constancia: float64 - 660 valores únicos
• score_recursos: float64 - 14 valores únicos
• categoria_rendimiento: str - 3 valores únicos
• interaccion_socioescuela: int64 - 5 valores únicos
• balance_vida_estudio: float64 - 50 valores únicos
